In [153]:
# Standard library imports
import ssl
import os
import importlib

# PyTorch imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

# Specialized ML imports
from transformers import get_scheduler

# Type hinting imports
from utils import set_seed

# Configure SSL for certain operations
ssl._create_default_https_context = ssl._create_unverified_context

from my_datasets import ObjectsDataset, CelebA, DSprites
import my_datasets
import importlib
import trainers

from models import VQELAgent
from trainers import train_self_play, train_mutual_play
from utils import load_dataset
from encoders import ProtoNetCNNEncoder, Dino

import torch.nn.functional as F
from torch.distributions import Categorical
from tqdm import tqdm

from test_time import test_time_adaptation
from test_time import test_time_scaling
from test_time import test_time_batch_adaptation
from utils import pairwise_cosine_similarity


from my_datasets import MNIST

## Config

In [154]:
config_path = os.getenv('CONFIG', "config.py")

if config_path and os.path.exists(config_path):
    spec = importlib.util.spec_from_file_location("config", config_path)
    CFG = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(CFG)
else:
    print("No config file found.")

In [155]:
from dataclasses import dataclass
from typing import Optional, Any
from datetime import datetime

@dataclass
class VQExperimentConfig:
    """Configuration class for VQEL experiment."""
    
    dataset_tt = CFG.dataset_tt
    num_iterations = CFG.num_iterations
    learning_rate_tt = CFG.learning_rate_tt
    message_length_tt = CFG.message_length_tt
    test_time_mode = CFG.test_time_mode
    sampling_temperature_tt = CFG.sampling_temperature_tt
    best_of_n = CFG.best_of_n
    
    pretrained_checkpoint_a = CFG.pretrained_checkpoint_a
    pretrained_checkpoint_b = CFG.pretrained_checkpoint_b
    dialogued_checkpoint = CFG.dialogued_checkpoint
    sim: str = CFG.sim
    dataset: str = CFG.dataset
        
    # Model hyperparameters
    batch_size: int = 32
    vocab_size: int = CFG.vocab_size
    representation_dim: int = CFG.representation_dim
    input_dim: int = 40
    message_length = CFG.message_length
    
    # VQ-specific hyperparameters
    ema_dead_code_factor = 2
    threshold_ema_dead_code: float = None  # batch_size / (vocab_size * ema_dead_code_factor)
    decay: float = CFG.decay
    commitment_weight: float = CFG.commitment_weight
    
    # Training hyperparameters
    learning_rate_phase1: float = CFG.learning_rate_phase1  # Learning rate for Phase 1 (Pretraining)
    learning_rate_phase2_a: float = CFG.learning_rate_phase2_a  # Learning rate for Phase 2 (Dialogue)
    learning_rate_phase2_b: float = CFG.learning_rate_phase2_b  # Learning rate for Phase 2 (Dialogue)

    weight_decay: float = 1e-5
    num_warmup_steps: int = 100
    
    # Pretraining parameters
    num_pretrain_epochs: int = CFG.num_pretrain_epochs
    
    # Dialogue training parameters
    num_dialogue_epochs: int = CFG.num_dialogue_epochs
    sampling_temperature: float = CFG.sampling_temperature
    entropy_regularization_factor: float = CFG.entropy_regularization_factor
    contrastive_loss_temperature: float = CFG.contrastive_loss_temperature
    agent_a_training_mode: str = CFG.agent_a_training_mode  # Options: 'frozen', 'reinforce_only', 'reinforce_with_preservation'
    freeze_codebook: bool = CFG.freeze_codebook  # Whether to freeze the codebook during text generation
    
    # Data split ratios
    train_split: float = 0.8
    val_split: float = 0.1
    test_split: float = 0.1
    
    # DataLoader parameters
    num_workers: int = 0
    drop_last: bool = True
    
    # Evaluation parameters
    number_of_candidates: int = CFG.number_of_candidates
    
    # System parameters
    seed: int = CFG.seed
    device: str = "auto"  # Will be set to 'cuda' or 'cpu' based on availability
    ckpt_dir: Optional[str] = None  # Will be generated from hyperparameters
    ckpt_dir_dialogue: Optional[str] = None  # Will be generated from hyperparameters
    
    # Logging parameters
    log_level: str = "INFO"
    tensorboard_log_dir: Optional[str] = None  # Will be generated from hyperparameters
    
    def __post_init__(self):
        """Post-initialization to set device and validate splits."""
        self.threshold_ema_dead_code: float =   self.batch_size / (self.vocab_size * self.ema_dead_code_factor)

        if self.device == "auto":
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        # Validate that splits sum to 1.0
        total_split = self.train_split + self.val_split + self.test_split
        if abs(total_split - 1.0) > 1e-6:
            raise ValueError(f"Data splits must sum to 1.0, got {total_split}")
        
        # Validate agent training mode
        valid_modes = ['frozen', 'reinforce_only', 'reinforce_with_preservation']
        if self.agent_a_training_mode not in valid_modes:
            raise ValueError(f"agent_a_training_mode must be one of {valid_modes}, got {self.agent_a_training_mode}")
        
        # Validate datasets
        valid_datasets = ['objects', 'shape', 'celeba', 'dsprites', 'mnist1']
        if self.dataset not in valid_datasets:
            raise ValueError(f"dataset must be one of {valid_datasets}, got {self.dataset}")
        
        # Generate checkpoint directories from hyperparameters
        if self.ckpt_dir is None or self.ckpt_dir_dialogue is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M")
            base_run_name = (
                f"{timestamp}_bs{self.batch_size}_vocab{self.vocab_size}_"
                f"repr{self.representation_dim}_"
                f"lr1_{self.learning_rate_phase1}_lr2a_{self.learning_rate_phase2_a}_lr2b_{self.learning_rate_phase2_b}_"
                f"decay{self.decay}_mode{self.agent_a_training_mode}_"
                f"temp{self.sampling_temperature}_ent{self.entropy_regularization_factor}_"
                f"cand{self.number_of_candidates}_"
                f"contr{self.contrastive_loss_temperature}_seed{self.seed}"
            )
            
            if self.ckpt_dir is None:
                self.ckpt_dir = f"runs/{base_run_name}/checkpoints/pretraining"
            
            if self.ckpt_dir_dialogue is None:
                self.ckpt_dir_dialogue = f"runs/{base_run_name}/checkpoints/dialogue"
        
        # Generate tensorboard log directory from hyperparameters
        if self.tensorboard_log_dir is None:
            # Extract the base run name from ckpt_dir (remove checkpoints/ and /pretraining)
            base_path = self.ckpt_dir.replace('/pretraining', '').replace('/checkpoints', '')
            self.tensorboard_log_dir = f"{base_path}/tensorboard"
    
    def to_dict(self) -> dict[str, Any]:
        """Convert config to dictionary."""
        return {
            'dataset_tt': self.dataset_tt,
            'test_time_mode': self.test_time_mode,
            'sampling_temperature_tt': self.sampling_temperature_tt,
            'best_of_n': self.best_of_n,
            'num_iterations': self.num_iterations,
            'learning_rate_tt': self.learning_rate_tt,
            'message_length_tt': self.message_length_tt,
            'pretrained_checkpoint_a': self.pretrained_checkpoint_a,
            'pretrained_checkpoint_b': self.pretrained_checkpoint_b,
            'dialogued_checkpoint': self.dialogued_checkpoint,
            'sim': self.sim,
            'dataset': self.dataset,
            'batch_size': self.batch_size,
            'vocab_size': self.vocab_size,
            'representation_dim': self.representation_dim,
            'input_dim': self.input_dim,
            'message_length': f"{self.message_length}",
            'threshold_ema_dead_code': self.threshold_ema_dead_code,
            'decay': self.decay,
            'commitment_weight': self.commitment_weight,
            'learning_rate_phase1': self.learning_rate_phase1,
            'learning_rate_phase2_a': self.learning_rate_phase2_a,
            'learning_rate_phase2_b': self.learning_rate_phase2_b,
            'weight_decay': self.weight_decay,
            'num_warmup_steps': self.num_warmup_steps,
            'num_pretrain_epochs': self.num_pretrain_epochs,
            'num_dialogue_epochs': self.num_dialogue_epochs,
            'sampling_temperature': self.sampling_temperature,
            'entropy_regularization_factor': self.entropy_regularization_factor,
            'contrastive_loss_temperature': self.contrastive_loss_temperature,
            'agent_a_training_mode': self.agent_a_training_mode,
            'freeze_codebook': self.freeze_codebook,
            'train_split': self.train_split,
            'val_split': self.val_split,
            'test_split': self.test_split,
            'num_workers': self.num_workers,
            'drop_last': self.drop_last,
            'number_of_candidates': self.number_of_candidates,
            'seed': self.seed,
            'device': self.device,
            'ckpt_dir': self.ckpt_dir,
            'ckpt_dir_dialogue': self.ckpt_dir_dialogue,
            'log_level': self.log_level,
            'tensorboard_log_dir': self.tensorboard_log_dir
        }

# Create experiment configuration
config = VQExperimentConfig()
print(f"VQEL experiment configuration created: {config}")


VQEL experiment configuration created: VQExperimentConfig(sim='cosine', dataset='mnist1', batch_size=32, vocab_size=10, representation_dim=192, input_dim=40, threshold_ema_dead_code=1.6, decay=0.99, commitment_weight=0.25, learning_rate_phase1=0.0001, learning_rate_phase2_a=1e-05, learning_rate_phase2_b=1e-05, weight_decay=1e-05, num_warmup_steps=100, num_pretrain_epochs=50, num_dialogue_epochs=50, sampling_temperature=1e-05, entropy_regularization_factor=0, contrastive_loss_temperature=0.01, agent_a_training_mode='frozen', freeze_codebook=True, train_split=0.8, val_split=0.1, test_split=0.1, num_workers=0, drop_last=True, number_of_candidates=100, seed=1, device='cuda', ckpt_dir='runs/20251224_2310_bs32_vocab10_repr192_lr1_0.0001_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1/checkpoints/pretraining', ckpt_dir_dialogue='runs/20251224_2310_bs32_vocab10_repr192_lr1_0.0001_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1

In [156]:
# Use config parameters instead of hardcoded values
device = torch.device(config.device)  # config.device is guaranteed to be set in __post_init__
train_batch_size = config.batch_size
representation_dim = config.representation_dim
vocab_size = config.vocab_size
message_length = config.message_length
threshold_ema_dead_code = config.threshold_ema_dead_code

In [157]:
# Initialize random seed for reproducibility
set_seed(config.seed)

# Create dataset and split into train/validation/test sets
if config.dataset == 'objects':
    dataset = ObjectsDataset()
    train_dataset, val_dataset, test_dataset = random_split(dataset, [config.train_split, config.val_split, config.test_split])
    
    object_encoder_a = nn.Sequential(
        nn.Linear(config.input_dim, config.representation_dim)
    )
    object_encoder_b = nn.Sequential(
        nn.Linear(config.input_dim, config.representation_dim)
    )
    
elif config.dataset == 'shape':
    train_dataset = load_dataset("/home/shared/data/shape/train")
    val_dataset = load_dataset("/home/shared/data/shape/val")
    test_dataset = load_dataset("/home/shared/data/shape/test/one_shape")
    if config.dataset_tt == 'one_shape':
        test_time_dataset = load_dataset("/home/shared/data/shape/test/one_shape")
    else:
        test_time_dataset = load_dataset("/home/shared/data/shape/test/two_shape")
    
    object_encoder_a = ProtoNetCNNEncoder()
    object_encoder_b = ProtoNetCNNEncoder()
elif config.dataset == 'celeba':
    dataset = CelebA(n=50000)
    train_dataset, val_dataset, test_dataset = random_split(dataset, [config.train_split, config.val_split, config.test_split])
    
    object_encoder_a = Dino()
    object_encoder_b = Dino()
    
elif config.dataset == 'dsprites':
    dataset = my_datasets.DSprites(n=20000)
    train_dataset, val_dataset, test_dataset = random_split(dataset, [config.train_split, config.val_split, config.test_split])
    
    object_encoder_a = ProtoNetCNNEncoder(in_channels=1)
    object_encoder_b = ProtoNetCNNEncoder(in_channels=1)
    
elif config.dataset == 'mnist1':
    dataset = MNIST('train', path='/home/shared/data/MNIST1')
    test_dataset = random_split(MNIST('test', path='/home/shared/data/MNIST1'), [2000, 8000])[0]
    
    train_dataset, val_dataset = random_split(dataset, [50000, 10000])
    
    object_encoder_a = ProtoNetCNNEncoder(in_channels=1)
    object_encoder_b = ProtoNetCNNEncoder(in_channels=1)
    
    if config.dataset_tt == 'mnist1':
        test_time_dataset = random_split(MNIST('test', path='/home/shared/data/MNIST1'), [2000, 8000])[0]
    else:
        test_time_dataset = random_split(MNIST('test', path='/home/shared/data/MNIST2'), [2000, 8000])[0]

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.batch_size, 
    shuffle=True, 
    num_workers=config.num_workers, 
    drop_last=config.drop_last
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=config.batch_size, 
    shuffle=True, 
    num_workers=config.num_workers, 
    drop_last=config.drop_last
)


agent_a = VQELAgent(
    input_dim=config.input_dim, 
    representation_dim=config.representation_dim, 
    threshold_ema_dead_code=config.threshold_ema_dead_code, 
    vocab_size=config.vocab_size,
    object_encoder=object_encoder_a,
    decay=config.decay, 
    commitment_weight=config.commitment_weight,
    orthogonal_reg_weight=0,
    use_cosine_sim = (config.sim == "cosine")
).to(device)

agent_b = VQELAgent(
    input_dim=config.input_dim, 
    representation_dim=config.representation_dim, 
    threshold_ema_dead_code=config.threshold_ema_dead_code, 
    vocab_size=config.vocab_size,
    object_encoder=object_encoder_b,
    use_cosine_sim = (config.sim == "cosine")
).to(device)

In [158]:
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# Create tensorboard writer with auto-generated directory structure
writer = SummaryWriter(log_dir=config.tensorboard_log_dir)
logger.info(f"TensorBoard writer created at: {config.tensorboard_log_dir}")
logger.info(f"Checkpoint directory (Phase 1): {config.ckpt_dir}")
logger.info(f"Checkpoint directory (Phase 2): {config.ckpt_dir_dialogue}")


2025-12-24 23:10:59,541 [INFO] TensorBoard writer created at: runs/20251224_2310_bs32_vocab10_repr192_lr1_0.0001_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1/tensorboard
2025-12-24 23:10:59,542 [INFO] Checkpoint directory (Phase 1): runs/20251224_2310_bs32_vocab10_repr192_lr1_0.0001_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1/checkpoints/pretraining
2025-12-24 23:10:59,543 [INFO] Checkpoint directory (Phase 2): runs/20251224_2310_bs32_vocab10_repr192_lr1_0.0001_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1/checkpoints/dialogue


In [159]:
if config.pretrained_checkpoint_b != None:
    agent_b.load_state_dict(torch.load(f'../experiments/{config.pretrained_checkpoint_b}/checkpoints/pretraining/best_model.pth', weights_only=True)['agent'])

if config.pretrained_checkpoint_a != None:
    agent_a.load_state_dict(torch.load(f'../experiments/{config.pretrained_checkpoint_a}/checkpoints/pretraining/best_model.pth', weights_only=True)['agent'])

if (config.pretrained_checkpoint_a == None) and (config.pretrained_checkpoint_b == None):
    # # Configure optimizer with weight decay for regularization (Phase 1)
    optimizer = torch.optim.Adam(
        agent_a.parameters(), 
        lr=config.learning_rate_phase1, 
        weight_decay=config.weight_decay
    )
    # 
    # Set up learning rate scheduler with warmup (Phase 1)
    lr_scheduler = get_scheduler(
        'constant_with_warmup', 
        optimizer=optimizer, 
        num_warmup_steps=config.num_warmup_steps
    )
    # 
    # Pretrain agent A with specified hyperparameters
    trainers.train_self_play(
        agent_a, 
        train_loader, 
        val_loader, 
        optimizer, 
        lr_scheduler, 
        device, 
        config.num_pretrain_epochs, 
        length_message=config.message_length, 
        sampling_temperature=config.sampling_temperature, 
        entropy_factor=int(config.entropy_regularization_factor), 
        contrastive_loss_temperature=config.contrastive_loss_temperature, 
        ckpt_dir=config.ckpt_dir,
    )

In [160]:
from utils import evaluate_self_communicate

self_play_acc_a = evaluate_self_communicate(
    agent_a, test_dataset, device, config.message_length[-1], number_of_candidates=config.number_of_candidates
)

Evaluating image-text matching:   0%|          | 0/20 [00:00<?, ?it/s]

Image-text matching accuracy: 0.788


In [161]:
self_play_acc_b = evaluate_self_communicate(
    agent_b, test_dataset, device, config.message_length[-1], number_of_candidates=config.number_of_candidates
)

Evaluating image-text matching:   0%|          | 0/20 [00:00<?, ?it/s]

Image-text matching accuracy: 0.009


# Play with Receiver 

In [162]:
# Set up optimizer for both agents with appropriate learning rate and weight decay (Phase 2)
optimizer = torch.optim.Adam(
    [
        {
            "params": agent_a.parameters(),
            "lr": config.learning_rate_phase2_a,
            "weight_decay": config.weight_decay,
        },
        {
            "params": agent_b.parameters(),
            "lr": config.learning_rate_phase2_b,
            "weight_decay": config.weight_decay,
        }
    ]
)


# Calculate training steps and warmup schedule
num_training_steps = config.num_pretrain_epochs * len(train_loader)

# Initialize learning rate scheduler with warmup (Phase 2)
lr_scheduler = get_scheduler(
    'constant_with_warmup', 
    optimizer=optimizer, 
    num_warmup_steps=config.num_warmup_steps,
)

In [163]:
if config.dialogued_checkpoint != None:
    agent_a.load_state_dict(torch.load(f'../experiments/{config.dialogued_checkpoint}/checkpoints/dialogue/best_model.pth', weights_only=True)['agent_a'])
    agent_b.load_state_dict(torch.load(f'../experiments/{config.dialogued_checkpoint}/checkpoints/dialogue/best_model.pth', weights_only=True)['agent_b'])
else:
    best_model_state, best_val_acc = trainers.train_mutual_play(
        agent_a, 
        agent_b, 
        train_loader, 
        val_loader, 
        device, 
        config.message_length, 
        config.num_pretrain_epochs, 
        optimizer, 
        lr_scheduler, 
        num_dialogue_epochs=config.num_dialogue_epochs,
        sampling_temperature=config.sampling_temperature,
        entropy_regularization_factor=config.entropy_regularization_factor,
        contrastive_loss_temperature=config.contrastive_loss_temperature,
        ckpt_dir=config.ckpt_dir_dialogue,
        agent_a_training_mode=config.agent_a_training_mode,
        freeze_codebook=config.freeze_codebook,
        tensorboard_writer=writer
    )
    
    agent_a.load_state_dict(torch.load(f'./{config.ckpt_dir_dialogue}/best_model.pth', weights_only=True)['agent_a'])
    agent_b.load_state_dict(torch.load(f'./{config.ckpt_dir_dialogue}/best_model.pth', weights_only=True)['agent_b'])

In [164]:
from utils import evaluate_cross_communicate

# Evaluate test accuracy
mutual_play_acc = evaluate_cross_communicate(agent_a, agent_b, test_dataset, device, config.message_length[-1], config.number_of_candidates)
print(mutual_play_acc)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

0.8955


# Test time Computation

In [165]:
# Accuracy on OOD without test time training or adaptation
evaluate_cross_communicate(agent_a, agent_b, test_time_dataset, device, config.message_length_tt, config.number_of_candidates)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

0.436

In [166]:
import test_time
importlib.reload(test_time)

<module 'test_time' from '/home/mehdi_jmlkh/test-time-emrgent-language/VQTT/src/test_time.py'>

In [167]:
test_loader = DataLoader(test_time_dataset, batch_size=config.number_of_candidates, shuffle=False, num_workers=0)

if config.test_time_mode == 'batch_adaptation':

    test_time_batch_adaptation(agent_a, test_loader, message_length=config.message_length_tt, num_epochs=config.num_iterations, lr=config.learning_rate_tt, sampling_temperature=config.sampling_temperature_tt)

    ttc_self_play_acc = evaluate_self_communicate(agent_a, test_time_dataset, device, config.message_length_tt, number_of_candidates=config.number_of_candidates)
    ttc_mutual_acc = evaluate_cross_communicate(agent_a, agent_b, test_time_dataset, device, config.message_length_tt, config.number_of_candidates)
else:
    total_correct, total_ins = 0, 0
    total_correct_a = 0

    agent_b.eval()
    for iter_num, (imgs, labels) in enumerate(tqdm(test_loader, desc="Testing batches")):
        imgs = imgs.to(device)

        ttc_words = []
        ttc_word_reprs = []
        for i in tqdm(range(len(imgs)), desc="Test-time training", leave=False):
            sample = imgs[i]

            if config.test_time_mode == 'scaling':
                ttc_word, ttc_discretized = test_time_scaling(
                    agent_a, sample, 
                    n=config.best_of_n, 
                    sampling_temperature=config.sampling_temperature_tt, 
                    message_length=config.message_length_tt
                )
            else:
                ttc_word, ttc_discretized = test_time_adaptation(
                    agent_a, sample, num_iterations=config.num_iterations, 
                    lr=config.learning_rate_tt, 
                    sampling_temperature=config.sampling_temperature_tt, 
                    length_message=config.message_length_tt
                )

            ttc_words.append(ttc_word)
            ttc_word_reprs.append(ttc_discretized)
        # Stack all words into a batch
        words = torch.stack(ttc_words, dim=0).squeeze(1)
        word_reprs = torch.stack(ttc_word_reprs, dim=0).squeeze(1)

        # Evaluate with the listener (agent_b)
        listener_messages_repr = agent_b.forward_external_text_perception(words).squeeze(1)
        listener_objects_repr = agent_b.forward_image_encoder(imgs)
        similarities_messages_to_objects = (F.normalize(listener_messages_repr, p=2, dim=1) @ F.normalize(listener_objects_repr, p=2, dim=1).T)
        batch_correct = (similarities_messages_to_objects.argmax(-1) == torch.arange(0, len(similarities_messages_to_objects)).to(device)).sum().item()
        total_correct += batch_correct
        total_ins += len(imgs)

        # Evaulate with the sender (agent_a)
        agent_a.eval()
        text_representations_a = agent_a.forward_text_perception(word_reprs)
        image_representations_a = agent_a.forward_image_encoder(imgs)
        similarity_matrix = pairwise_cosine_similarity(text_representations_a, image_representations_a)
        predicted_matches = similarity_matrix.argmax(dim=1).cpu()
        total_correct_a += (predicted_matches == torch.arange(imgs.shape[0])).sum().item()

        # Show accuracy after each batch
        batch_accuracy = round(batch_correct/len(imgs), 3)
        cumulative_accuracy = round(total_correct/total_ins, 3)
        print(f'Batch {iter_num+1} accuracy: {batch_accuracy}, Cumulative accuracy: {cumulative_accuracy}')

        agent_a.train()

    ttc_self_play_acc = total_correct_a / total_ins
    ttc_mutual_acc = total_correct / total_ins


print('Final test accuracy with test-time training:', round(ttc_mutual_acc, 3))
print('Final self play accuracy:', round(ttc_self_play_acc, 3))

Training Progress Epoch 0/10:   0%|          | 0/20 [00:00<?, ?it/s]

Training Progress Epoch 1/10:   0%|          | 0/20 [00:00<?, ?it/s]

Training Progress Epoch 2/10:   0%|          | 0/20 [00:00<?, ?it/s]

Training Progress Epoch 3/10:   0%|          | 0/20 [00:00<?, ?it/s]

Training Progress Epoch 4/10:   0%|          | 0/20 [00:00<?, ?it/s]

...

Training Progress Epoch 9/10:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating image-text matching:   0%|          | 0/20 [00:00<?, ?it/s]

Image-text matching accuracy: 0.635


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Final test accuracy with test-time training: 0.56
Final self play accuracy: 0.635


## Save

In [168]:
import json
import os

result_dir = config.tensorboard_log_dir.replace("tensorboard", "results")
os.makedirs(result_dir, exist_ok=True)

results = {
    "VQEL": True,
    "metrics": {
        "self_play_accuracy_a": round(self_play_acc_a, 3),
        "self_play_accuracy_b": round(self_play_acc_b, 3),
        "mutual_play_accuracy": round(mutual_play_acc, 3),
        "test_time_self_play_accuracy": round(ttc_self_play_acc, 3),
        "test_time_training_accuracy": round(ttc_mutual_acc, 3)
    },
    "config": {k: v for k, v in config.to_dict().items()}
}

with open(os.path.join(result_dir, "report.json"), "w") as file:
    json.dump(results, file, indent=4)